# Experiment C - Reconstruction-Trained Acceleration CNN

**Question:** how much character-discriminative information is recoverable when the classifier is trained directly on reconstructed acceleration?

Strict protocol:

1. Reuse Experiment A's **train / validation / test users** and `class_to_idx` from the A checkpoint.
2. Use the A checkpoint **only for split/class metadata**. Do **not** load A model weights.
3. Fit acceleration normalization using **reconstruction train valid samples only**.
4. Train a new `MaskAwareAccelerationCNN` from scratch on reconstruction train.
5. Select the best epoch using reconstruction validation balanced accuracy.
6. Evaluate on reconstruction test with the shared representation protocol.
7. Use the same selected dataset roots as the A checkpoint; changing the action/sample cohort requires regenerating A.

This notebook is intentionally thin. The executable experiment lives in `scripts/run_experiment_c.py`; the reusable implementation lives in `snn/accel_reconstruction_eval/`.


In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
from pathlib import Path
import sys

import pandas as pd
import torch


def find_repository_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "snn").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate repository root containing the 'snn' directory."
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd())
SCRIPTS_DIR = REPOSITORY_ROOT / "scripts"

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from snn.accel_reconstruction_eval import experiment_c_config
from run_experiment_c import run_experiment_c

print("Repository root:", REPOSITORY_ROOT)
print("CUDA available:", torch.cuda.is_available())


## Paths and experiment controls

The reference checkpoint is the Experiment A checkpoint. It is used only to recover the exact user split and class mapping. Experiment C does not reuse its weights or normalization statistics.

Regenerate Experiment A before running B/C/D whenever A's exclusions change. C keeps strict reference mode and has no local cohort or explicit-split override.


In [ ]:
# Select the same one or two roots used to produce REFERENCE_A_CHECKPOINT.
# Roots remain separate on disk and are combined only in the shared loader.
DATASET_ROOTS = [
    Path("outputs/action0_rectified/low-pass/aligned-board-events"),
    Path("outputs/action1_rectified/low-pass/aligned-board-events"),
]
REFERENCE_A_CHECKPOINT = Path(
    "notebooks/artifacts/acceleration_cnn_representation/"
    "best_acceleration_cnn.pt"
)
OUTPUT_DIR = Path(
    "notebooks/artifacts/experiment_C_reconstruction_trained"
)

# Keep this False for strict A/B/C comparability.
ALLOW_NEW_SPLIT = False

# Optional training overrides. These defaults match the current CNN baseline.
NUM_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 10
BATCH_SIZE = 128
NUM_WORKERS = 0
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.0
USE_CLASS_WEIGHTS = False
GRAD_CLIP_NORM = None
MAX_TRAIN_BATCHES = None
USE_GPU = True


## Build the Experiment C configuration

Only the training/runtime settings below are modified. The Experiment C domain contract remains fixed as:

- train source = reconstruction
- validation source = reconstruction
- test source = reconstruction
- normalization source = reconstruction train


In [ ]:
config = experiment_c_config(
    output_dir=OUTPUT_DIR,
    random_seed=12345,
)

config = replace(
    config,
    use_gpu=USE_GPU,
    loader=replace(
        config.loader,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    ),
    training=replace(
        config.training,
        num_epochs=NUM_EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        use_class_weights=USE_CLASS_WEIGHTS,
        grad_clip_norm=GRAD_CLIP_NORM,
        max_train_batches=MAX_TRAIN_BATCHES,
    ),
)
config.validate()
config.to_dict()


## Run Experiment C

This call performs the complete pipeline:

`load -> recover A split -> fit recon normalization -> train from scratch -> select best epoch -> extract embeddings -> evaluate -> save artifacts`


In [ ]:
run = run_experiment_c(
    root=DATASET_ROOTS,
    repository_root=REPOSITORY_ROOT,
    output_dir=OUTPUT_DIR,
    reference_checkpoint=REFERENCE_A_CHECKPOINT,
    config=config,
    allow_new_split=ALLOW_NEW_SPLIT,
)


## Inherited cohort provenance

Experiment C inherits the cohort from the authoritative A checkpoint.


In [ ]:
provenance = json.loads(
    (run.output_dir / "provenance.json").read_text(encoding="utf-8")
)
display(
    pd.Series(
        {
            "cohort_source": provenance["cohort_source"],
            "excluded_users": provenance["excluded_users"],
            "eligible_users": provenance["eligible_users"],
            "train_users": provenance["train_users"],
            "val_users": provenance["val_users"],
            "test_users": provenance["test_users"],
            "class_to_idx": provenance["class_to_idx"],
        },
        name="Inherited cohort",
    )
)


## Primary results

In [ ]:
display(run.summary.T.rename(columns={0: "Experiment C"}))


## CNN classification across splits

In [ ]:
display(run.classification_splits)


## Split contract

In [ ]:
display(run.split_summary)
display(run.label_split_counts)


## Reconstruction-train normalization

In [ ]:
pd.DataFrame(
    {
        "channel": ["x", "y", "z"],
        "mean": run.normalization.mean,
        "std": run.normalization.std,
    }
).assign(
    fitted_on=run.normalization.fitted_on,
    valid_time_points=run.normalization.valid_time_points,
)


## Training history

In [ ]:
display(run.training_history.tail(20))


## Saved artifacts

In [ ]:
artifact_table = pd.DataFrame(
    [
        {"artifact": name, "path": str(path)}
        for name, path in sorted(run.artifact_paths.items())
    ]
)
display(artifact_table)


# Visualization

The cells below are presentation-only. They consume the metrics and embeddings already generated by `run_experiment_c()`.

In [ ]:
from snn.accel_reconstruction_eval.visualization import visualize_experiment_c

figures = visualize_experiment_c(run.output_dir)

## How to interpret C together with A and B

The key comparison is not B alone:

- **A high, B low, C approximately A:** reconstruction still contains task information; the main problem is raw-to-reconstruction domain shift.
- **A high, B low, C also low:** reconstruction has likely removed task-relevant information.
- **C > A:** reconstruction may provide a useful denoising or bottleneck effect; confirm this with geometry/retrieval metrics and repeated seeds before making a strong claim.

The final A/B/C comparison should use the same user split, class mapping, CNN architecture, and metric implementation.
